# GRID-Planner nuBoard Viewer

Use this notebook to open nuPlan closed-loop simulation results in `nuBoard`.

Workflow:
1. Run a closed-loop simulation first so that a result directory containing `.nuboard` files is generated.
2. Fill in the paths in the configuration cell below.
3. Execute the notebook cells in order.

Important:
- `RESULT_FOLDER` should point to a simulation output directory, not a training checkpoint directory.
- Set `SCENARIO_BUILDER` to match the split used during simulation.
  - Use `nuplan` for `val14`
  - Use `nuplan_challenge` for challenge/test splits such as `test14-hard` and `test14-random`
- If port `6599` is occupied, change `PORT_NUMBER` below.


In [ ]:
import os
import sys
from pathlib import Path

import hydra


## User Configuration


In [ ]:
RESULT_FOLDER = "REPLACE_WITH_RESULT_PATH"  # e.g. /home/results/.../exp/simulation/closed_loop_reactive_agents/flow_planner/val14/...
SCENARIO_BUILDER = "nuplan"  # use "nuplan_challenge" for challenge/test splits
PORT_NUMBER = 6599
PROJECT_ROOT = str(Path.cwd().resolve())

env_variables = {
    "NUPLAN_DEVKIT_ROOT": "REPLACE_WITH_NUPLAN_DEVKIT_DIR",
    "NUPLAN_DATA_ROOT": "REPLACE_WITH_DATA_DIR",
    "NUPLAN_MAPS_ROOT": "REPLACE_WITH_MAPS_DIR",
    "NUPLAN_EXP_ROOT": "REPLACE_WITH_EXP_DIR",
    "NUPLAN_SIMULATION_ALLOW_ANY_BUILDER": "1",
}

for key, value in env_variables.items():
    os.environ[key] = value

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{os.environ.get('PYTHONPATH', '')}"

NUBOARD_CONFIG_DIR = str(
    Path(os.environ["NUPLAN_DEVKIT_ROOT"]) / "nuplan" / "planning" / "script" / "config" / "nuboard"
)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RESULT_FOLDER =", RESULT_FOLDER)
print("NUBOARD_CONFIG_DIR =", NUBOARD_CONFIG_DIR)


In [ ]:
result_root = Path(RESULT_FOLDER)
if not result_root.exists():
    raise FileNotFoundError(f"RESULT_FOLDER not found: {result_root}")

config_dir = Path(NUBOARD_CONFIG_DIR)
if not config_dir.exists():
    raise FileNotFoundError(f"nuBoard config dir not found: {config_dir}")

nuboard_files = sorted(result_root.rglob("*.nuboard"))
if not nuboard_files:
    raise FileNotFoundError(
        f"No .nuboard files found under {result_root}. Please make sure closed-loop simulation finished successfully."
    )

simulation_paths = sorted({str(path.parent) for path in nuboard_files})
print(f"Found {len(nuboard_files)} .nuboard file(s) under {result_root}.")
for path in simulation_paths:
    print(" -", path)

hydra.core.global_hydra.GlobalHydra.instance().clear()
hydra.initialize_config_dir(config_dir=str(config_dir), version_base=None)

cfg = hydra.compose(
    config_name="default_nuboard",
    overrides=[
        f"scenario_builder={SCENARIO_BUILDER}",
        f"simulation_path={simulation_paths}",
        "hydra.searchpath=[pkg://flow_planner.nuplan_simulation.scenario_filter, pkg://flow_planner.nuplan_simulation, pkg://nuplan.planning.script.config.common, pkg://nuplan.planning.script.experiments]",
        f"port_number={PORT_NUMBER}",
    ],
)

cfg


In [ ]:
from nuplan.planning.script.run_nuboard import main as main_nuboard

main_nuboard(cfg)
